# v26 — Filter Exhaustion Multi-Timeframe (RSI M15/M30/H4/D1)

**Latar belakang:** Investigasi 3 loss beruntun terakhir (21 Agustus 2026, diverifikasi ulang
langsung dari MT5 HFM) menemukan pola konsisten: robot v13 entry BUY tepat setelah rally cepat
yang membuat RSI **overbought (>68) BERSAMAAN di SEMUA timeframe** -- M15 (RSI 68-81), M30
(RSI 73-78), H4 (RSI 70.6), D1 (RSI 77.0). v13 saat ini cuma cek EMA H1 utk arah trend
(alignment), TIDAK ADA pengecekan level overbought/oversold di timeframe manapun.

**v25 sudah membuktikan** ADX tinggi SAJA bukan indikator yang berguna (grid search 420
kombinasi ADX ceiling/exhaustion gagal total). Tapi ADX cuma ukur KEKUATAN trend, bukan
SEBERAPA JAUH harga sudah bergerak/jenuh -- RSI mengukur hal yang beda (momentum relatif thd
range terakhir), dan mengeceknya di TIMEFRAME LEBIH TINGGI (bukan cuma M5) berpotensi
menangkap "rally sudah kepanjangan" yang tidak kelihatan dari M5 sendirian.

**Cakupan v26**: filter TAMBAHAN di atas v13 yang sudah ada (bukan strategi baru) -- skip
entry kalau RSI M15/M30/H4/D1 (atau kombinasi) sudah di atas/bawah threshold overbought/
oversold saat entry BUY/SELL. Grid search threshold RSI & kombinasi timeframe mana yang paling
efektif, TERMASUK cek eksplisit apakah filter ini salah buang trade WIN yang valid (bukan cuma
menyaring LOSS).

**Metodologi**: sama persis pola v19-v25 -- TRAIN (2019-2023)/TEST (2024-2026) split, kriteria
kejujuran (kandidat harus MENGUNGGULI baseline v13 murni PF di TRAIN *dan* TEST, bukan cuma
PF>1.5 absolut krn baseline TRAIN sendiri di bawah itu -- lihat v25), spread real 1.82,
v12_score ASLI + Order Block filter + H1 alignment (BUKAN reimplementasi).

**TIDAK ADA perubahan ke `usecase.py`** -- murni riset backtest.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v26"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v23 (skor v12 + OB + H1 EMA) + gabung RSI M15/M30/H4/D1

In [2]:
MTF_RSI_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2019_2026_mtf_rsi.parquet"

if MTF_RSI_CACHE_PATH.exists():
    print(f"Load dari cache: {MTF_RSI_CACHE_PATH}")
    df = pd.read_parquet(MTF_RSI_CACHE_PATH)
else:
    print("Belum ada cache -- load v23 base + merge RSI M15/M30/H4/D1...")
    v23_cache = PROCESSED_DIR / "v23" / "df_2019_2026_full_mtf.parquet"
    assert v23_cache.exists(), "Cache v23 belum ada"
    df = pd.read_parquet(v23_cache)

    # Merge RSI + bull_chain/bear_chain dari tiap timeframe -- pakai merge_asof direction=backward
    # supaya cuma pakai candle timeframe tinggi yang SUDAH CLOSE sebelum candle M5 saat ini
    # (hindari lookahead, sama pola dgn merge H1 yang sudah ada).
    tf_configs = [
        ("m15", pd.Timedelta(minutes=15), "m15"),
        ("m30", pd.Timedelta(minutes=30), "m30"),
        ("h4", pd.Timedelta(hours=4), "h4"),
        ("d1", pd.Timedelta(days=1), "d1"),
    ]
    for tf_file, tf_delta, prefix in tf_configs:
        df_tf = pd.read_csv(
            PROCESSED_DIR / "v01" / f"xauusd_{tf_file}_full_indicators.csv",
            usecols=["datetime", "rsi", "adx", "bull_chain", "bear_chain"],
        )
        df_tf["datetime"] = pd.to_datetime(df_tf["datetime"])
        df_tf = df_tf.sort_values("datetime").reset_index(drop=True)
        df_tf["available_at"] = df_tf["datetime"] + tf_delta
        df_tf = df_tf.rename(columns={
            "rsi": f"{prefix}_rsi", "adx": f"{prefix}_adx",
            "bull_chain": f"{prefix}_bull_chain", "bear_chain": f"{prefix}_bear_chain",
        })
        cols_to_merge = ["available_at", f"{prefix}_rsi", f"{prefix}_adx", f"{prefix}_bull_chain", f"{prefix}_bear_chain"]
        df = pd.merge_asof(
            df.sort_values("datetime"), df_tf[cols_to_merge].sort_values("available_at"),
            left_on="datetime", right_on="available_at", direction="backward",
        )
        df = df.drop(columns=["available_at"])
        print(f"  merged {prefix}")

    df.to_parquet(MTF_RSI_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {MTF_RSI_CACHE_PATH}")

print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom: {list(df.columns)}")

Belum ada cache -- load v23 base + merge RSI M15/M30/H4/D1...


  merged m15


  merged m30


  merged h4
  merged d1


Tersimpan ke cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v26\df_2019_2026_mtf_rsi.parquet

Total candle: 518403, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom: ['datetime', 'open', 'high', 'low', 'close', 'adx', 'atr', 'v12_score', 'bull_chain', 'bear_chain', 'bos_bull', 'bos_bear', 'ob_bull', 'ob_bear', 'h1_ob_bull', 'h1_ob_bear', 'h1_ema_50', 'h1_ema_200', 'h1_bos_bull', 'h1_bos_bear', 'm15_rsi', 'm15_adx', 'm15_bull_chain', 'm15_bear_chain', 'm30_rsi', 'm30_adx', 'm30_bull_chain', 'm30_bear_chain', 'h4_rsi', 'h4_adx', 'h4_bull_chain', 'h4_bear_chain', 'd1_rsi', 'd1_adx', 'd1_bull_chain', 'd1_bear_chain']


## 2. Backtest engine v26: v13 + filter exhaustion multi-timeframe (RSI)

Filter: skip BUY kalau RSI di timeframe X >= `rsi_overbought_threshold`; skip SELL kalau RSI
di timeframe X <= `rsi_oversold_threshold` (simetris, `100 - overbought`). Bisa dipilih
kombinasi timeframe mana yang dicek (`check_m15`, `check_m30`, `check_h4`, `check_d1`) --
kalau BEBERAPA aktif, filter jalan kalau SALAH SATU dari timeframe yang dicek overbought/
oversold (AND logic tidak dipakai dulu, biar filter tidak terlalu longgar; bisa dicoba variasi
lain kalau perlu).

In [3]:
def check_h1_alignment_v26(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v26(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    rsi_overbought_threshold: float = 100.0,  # 100 = filter mati (tanpa syarat)
    check_m15: bool = False,
    check_m30: bool = False,
    check_h4: bool = False,
    check_d1: bool = False,
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()

    rsi_oversold_threshold = 100.0 - rsi_overbought_threshold if rsi_overbought_threshold < 100.0 else 0.0
    tf_rsi_arrs = {}
    for tf, enabled in [("m15", check_m15), ("m30", check_m30), ("h4", check_h4), ("d1", check_d1)]:
        if enabled:
            tf_rsi_arrs[tf] = df_signals[f"{tf}_rsi"].to_numpy()

    n = len(df_signals)
    trades = []
    equity = INITIAL_EQUITY
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v26(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        # Filter exhaustion multi-timeframe: skip kalau SALAH SATU timeframe yg dicek overbought/oversold
        if rsi_overbought_threshold < 100.0 and tf_rsi_arrs:
            exhausted = False
            for tf_rsi in tf_rsi_arrs.values():
                rsi_val = tf_rsi[i]
                if not np.isfinite(rsi_val):
                    continue
                if direction == "BUY" and rsi_val >= rsi_overbought_threshold:
                    exhausted = True
                    break
                if direction == "SELL" and rsi_val <= rsi_oversold_threshold:
                    exhausted = True
                    break
            if exhausted:
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        risk_amount = equity * RISK_PCT
        lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v26 siap.")

Backtest engine v26 siap.


## 3. TRAIN/TEST split & Baseline (v13 murni, tanpa filter RSI MTF)

In [4]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_base_train = run_backtest_v26(df_train)
trades_base_test = run_backtest_v26(df_test)
baseline_train = evaluate(trades_base_train, INITIAL_EQUITY)
baseline_test = evaluate(trades_base_test, INITIAL_EQUITY)
print("=== Baseline: v13 murni (tanpa filter RSI MTF) ===")
print("TRAIN:", baseline_train)
print("TEST :", baseline_test)

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle


=== Baseline: v13 murni (tanpa filter RSI MTF) ===
TRAIN: {'total_trades': 2282, 'win_rate_pct': 15.69, 'profit_factor': np.float64(0.38), 'net_pnl': np.float64(-2348.28), 'max_drawdown_pct': np.float64(-2348.28)}
TEST : {'total_trades': 1105, 'win_rate_pct': 45.7, 'profit_factor': np.float64(1.58), 'net_pnl': np.float64(1588.28), 'max_drawdown_pct': np.float64(-229.55)}


## 4. Cek dulu: apakah trade WIN historis JUGA sering py RSI MTF overbought/oversold?

Sebelum grid search, verifikasi kekhawatiran user -- apakah filter ini bakal salah buang
trade WIN yang valid juga, bukan cuma menyaring LOSS. Bandingkan distribusi RSI M15/M30/H4/D1
saat entry, dipecah WIN vs LOSS, di data TRAIN.

In [5]:
# Re-run baseline TAPI simpan RSI MTF per trade utk analisis (butuh join balik ke df_train by entry_time)
trades_base_train_idx = trades_base_train.copy()
trades_base_train_idx["entry_time"] = pd.to_datetime(trades_base_train_idx["entry_time"])
df_train_lookup = df_train[["datetime", "m15_rsi", "m30_rsi", "h4_rsi", "d1_rsi"]].rename(columns={"datetime": "entry_time"})
trades_with_rsi = trades_base_train_idx.merge(df_train_lookup, on="entry_time", how="left")

# Utk BUY, "exhausted" = RSI tinggi; utk SELL, "exhausted" = RSI rendah -- normalisasi jadi "rsi_in_entry_direction"
# supaya WIN vs LOSS bisa dibandingkan apple-to-apple terlepas arah
for tf in ["m15", "m30", "h4", "d1"]:
    trades_with_rsi[f"{tf}_rsi_directional"] = np.where(
        trades_with_rsi["direction"] == "BUY", trades_with_rsi[f"{tf}_rsi"], 100 - trades_with_rsi[f"{tf}_rsi"]
    )

print("=== RSI (arah-disesuaikan, makin tinggi = makin 'exhausted' searah entry) WIN vs LOSS ===")
for tf in ["m15", "m30", "h4", "d1"]:
    col = f"{tf}_rsi_directional"
    win_mean = trades_with_rsi[trades_with_rsi["result"]=="WIN"][col].mean()
    loss_mean = trades_with_rsi[trades_with_rsi["result"]=="LOSS"][col].mean()
    print(f"{tf.upper()}: WIN mean={win_mean:.2f}, LOSS mean={loss_mean:.2f}, diff={loss_mean-win_mean:+.2f}")

=== RSI (arah-disesuaikan, makin tinggi = makin 'exhausted' searah entry) WIN vs LOSS ===
M15: WIN mean=60.96, LOSS mean=60.21, diff=-0.75
M30: WIN mean=59.61, LOSS mean=58.00, diff=-1.61
H4: WIN mean=61.04, LOSS mean=59.08, diff=-1.96
D1: WIN mean=60.18, LOSS mean=59.62, diff=-0.56


## 5. Grid search: threshold RSI overbought x kombinasi timeframe

In [6]:
import itertools
import time as _time

GRID = {
    "rsi_overbought_threshold": [65.0, 70.0, 75.0, 80.0],
    "check_m15": [False, True],
    "check_m30": [False, True],
    "check_h4": [False, True],
    "check_d1": [False, True],
}

combos = list(itertools.product(*GRID.values()))
# Skip kombinasi tanpa timeframe aktif sama sekali (filter mati, sama dgn baseline)
combos = [c for c in combos if any(c[1:])]
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_v26(df_train, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 20 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 20 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(20).to_string(index=False))

print(f"\nBaseline TRAIN: PF={baseline_train['profit_factor']}, net_pnl={baseline_train['net_pnl']}, n={baseline_train['total_trades']}")
beating = grid_valid[grid_valid["profit_factor"] > baseline_train["profit_factor"]]
print(f"Kandidat mengungguli baseline TRAIN (PF lebih tinggi): {len(beating)} dari {len(grid_valid)}")

Total kombinasi grid: 60


  [20/60] 17s


  [40/60] 34s


  [60/60] 51s

Grid search selesai dalam 51s

=== Top 20 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct  rsi_overbought_threshold  check_m15  check_m30  check_h4  check_d1
         2136         15.07           0.39 -2152.44          -2152.44                      80.0       True       True     False      True
         1979         15.16           0.38 -2074.40          -2074.40                      70.0      False      False     False      True
         2220         15.50           0.38 -2258.76          -2258.76                      80.0       True      False     False     False
         2105         15.25           0.38 -2155.55          -2155.55                      80.0       True      False      True      True
         2129         15.27           0.38 -2173.89          -2173.89                      75.0       True      False     False     False
         1998         15.32           0.38 -2044.16          -2044.16            

## 6. Validasi TEST out-of-sample (kandidat yang mengungguli baseline TRAIN)

In [7]:
candidates_passing = beating.head(20)
print(f"Kandidat TRAIN mengungguli baseline: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat mengungguli baseline di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_results = []
    for _, row in candidates_passing.iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v26(df_test, **params)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                              "test_maxdd": m_test["max_drawdown_pct"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk kandidat yang menang di TRAIN ===")
    print(test_df.to_string(index=False))

    print(f"\nBaseline TEST: PF={baseline_test['profit_factor']}, net_pnl={baseline_test['net_pnl']}, "
          f"max_dd={baseline_test['max_drawdown_pct']}, n={baseline_test['total_trades']}")

    robust = test_df[(test_df["test_pf"] > baseline_test["profit_factor"]) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN mengungguli baseline: 1



=== Validasi TEST utk kandidat yang menang di TRAIN ===
 rsi_overbought_threshold  check_m15  check_m30  check_h4  check_d1  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
                     80.0       True       True     False      True      0.39     2136      1.6     986    45.84      1362.04     -192.42

Baseline TEST: PF=1.58, net_pnl=1588.28, max_dd=-229.55, n=1105

>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>=15): 1
 rsi_overbought_threshold  check_m15  check_m30  check_h4  check_d1  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
                     80.0       True       True     False      True      0.39     2136      1.6     986    45.84      1362.04     -192.42


## 7. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-6 -- placeholder)*